# Visualisasi Distribusi Uncertainty Score

Skrip ini dibuat berdasarkan instruksi `plan_generate_diagram.md` untuk memvisualisasikan grafik distribusi *Normalized Shannon Entropy* dan melihat titik ekuilibrium (Threshold) toleransi halusinasi pada model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

# 1. Definisikan path relatif ke 3 file JSON
# Sesuai dengan folder experiment/uncertainty/result/
base_dir = os.path.join("experiment", "uncertainty", "result")
files = {
    "Clear Data": os.path.join(base_dir, "scored_responses_clear_data.json"),
    "Aleatoric Data": os.path.join(base_dir, "scored_responses_aleatoric_data.json"),
    "Epistemic Data": os.path.join(base_dir, "scored_responses_epistemic_data.json")
}

# 2. Fungsi membaca JSON dan mengekstrak `normalized_uncertainty_score`
def load_data(filepath, label):
    scores = []
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for item in data:
                score = item.get("normalized_uncertainty_score")
                if score is not None:
                    scores.append(score)
    else:
        print(f"Peringatan: File {filepath} tidak ditemukan.")
    
    # Kembalikan sebagai DataFrame
    return pd.DataFrame({"score": scores, "type": label})

# 3. Masukkan data ke dalam satu Pandas DataFrame besar
df_list = []
for label, path in files.items():
    df_list.append(load_data(path, label))

df = pd.concat(df_list, ignore_index=True)

if df.empty:
    print("Tidak ada data untuk divisualisasikan. Pastikan path file JSON sudah benar!")
else:
    # 4. Lakukan visualisasi menggunakan seaborn
    # Mengatur style dan ukuran gambar
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(12, 7))
    
    # Pewarnaan sesuai konsep (Hijau, Oranye/Merah, Biru)
    palette_colors = {
        "Clear Data": "#2ecc71",        
        "Aleatoric Data": "#e67e22",    
        "Epistemic Data": "#3498db"     
    }
    
    # Multiple KDE Plot (Kernel Density Estimate) 
    # Agar overlapping (tumpang tindih) terlihat lebih mulus dan estetik dibandingkan histogram kasar
    sns.kdeplot(
        data=df,
        x="score",
        hue="type",
        fill=True,
        common_norm=False,
        palette=palette_colors,
        alpha=0.4,
        linewidth=2.5,
        warn_singular=False # Mencegah warning jika nilai seragam (semua 0)
    )
    
    # Set Batas Sumbu X (0.0 sampai 1.0) sesuai instruksi
    plt.xlim(0.0, 1.0)
    
    # Penyesuaian judul dan label sumbu
    plt.title("Distribusi Normalized Shannon Entropy (Uncertainty Score)", fontsize=16, fontweight='bold', pad=20)
    plt.xlabel("Uncertainty Score (Normalized)", fontsize=13)
    plt.ylabel("Data Density / Question Frequency", fontsize=13)
    
    # Dapatkan batas atas Y-axis saat ini untuk menempatkan teks anotasi secara dinamis/proporsional
    y_max = plt.ylim()[1]
    # Jika semua data 0, batas y mungkin sangat kecil, buat batas min logic
    if y_max < 0.1: 
        y_max = 1.0
        plt.ylim(0.0, 1.0)
    
    # 5. Garis Ambang Batas (Threshold Line)
    threshold_x = 0.400
    plt.axvline(x=threshold_x, color='black', linestyle='--', linewidth=2.5, label=f"Clarification Threshold (τ = {threshold_x})")
    
    # Anotasi teks Threshold di dekat garis
    plt.text(threshold_x + 0.015, y_max * 0.85, f"Clarification Threshold (τ = {threshold_x})", 
             color='black', fontsize=11, fontweight='bold', 
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray', boxstyle='round,pad=0.5'))
    
    # 6. Bayangan latar belakang (Background Span)
    # Area 0.0 - 0.400 (SYSTEM IS CONFIDENT)
    plt.axvspan(0.0, threshold_x, color='green', alpha=0.08)
    plt.text(threshold_x / 2, y_max * 0.95, "SYSTEM IS CONFIDENT", 
             color='green', fontsize=13, fontweight='bold', ha='center', alpha=0.8)
    
    # Area 0.400 - 1.0 (SYSTEM IS UNCERTAIN)
    plt.axvspan(threshold_x, 1.0, color='red', alpha=0.08)
    plt.text((1.0 + threshold_x) / 2, y_max * 0.95, "SYSTEM IS UNCERTAIN", 
             color='red', fontsize=13, fontweight='bold', ha='center', alpha=0.8)
    
    # Penyesuaian Legend
    plt.legend(title="Kelompok Data (Type)", loc='center right', bbox_to_anchor=(0.98, 0.75), fontsize=11, title_fontsize=12)
    
    # 7. Tampilkan Plot
    plt.tight_layout()
    plt.show()
